## **Step 1: Define the tools**

In [41]:
from langchain_openai import ChatOpenAI
import os 
from dotenv import load_dotenv
load_dotenv()
llm = ChatOpenAI(model="gpt-5-mini")

In [42]:
from langchain.tools import tool

In [43]:
@tool
def tool_duckduckgo_search(query: str) -> str:
    """Use this tool to gather information about current events or general knowledge"""

    from langchain_community.tools import DuckDuckGoSearchRun
    search = DuckDuckGoSearchRun()
    
    response = search.invoke(query)
    return response

    
tool_duckduckgo_search.invoke("What is the capital of France?")


'...ofthese citiesisthecapital, keep reading to find out. ...WhatisthecapitalcityofFrance?Thecapital, and largest, cityofFranceisParis. Besides Paris,whatisthecapitalofFrance? ... You use it between your head and your toes,themore it worksthethinner it grows. WhatistheCapitalofFrance? Paris ... Paris,thecapitalcityofFrance,isoneofthemost famous and influential cities intheworld. WhatistheCapitalofFrance? ... AsthecapitalcityofFrance,thecity plays host tothenational governmentofFrance. However, Paris only becametheofficialcapitalofFranceduringthereignofClovisI, inthelate 5th and early 6th century.'

In [44]:
@tool
def tool_wikipedia_search(query: str) -> str:
    """Use this tool to gather information about historical events"""

    from langchain_community.tools import WikipediaQueryRun
    from langchain_community.utilities import WikipediaAPIWrapper
    wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

    response = wikipedia.invoke(query)
    return response
    
tool_wikipedia_search.invoke("Alan Turing")


'Page: Alan Turing\nSummary: Alan Mathison Turing (; 23 June 1912 – 7 June 1954) was an English mathematician, computer scientist, logician, cryptanalyst, philosopher and theoretical biologist. He was highly influential in the development of theoretical computer science, providing a formalisation of the concepts of algorithm and computation with the Turing machine, which can be considered a model of a general-purpose computer. Turing is widely considered to be the father of theoretical computer science.\nBorn in London, Turing was raised in southern England. He graduated from King\'s College, Cambridge, and in 1938, earned a doctorate degree from Princeton University. During World War II, Turing worked for the Government Code and Cypher School at Bletchley Park, Britain\'s codebreaking centre that produced Ultra intelligence. He led Hut 8, the section responsible for German naval cryptanalysis. Turing devised techniques for speeding the breaking of German ciphers, including improvement

In [ ]:
@tool
def tool_arxiv_search(query: str) -> str:
    """Use this tool to gather information about arXiv papers"""

    from langchain_community.tools import ArxivQueryRun
    from langchain_community.utilities import ArxivAPIWrapper

    #1. Initialize the ArxivAPIWrapper
    arxiv_api_wrapper = ArxivAPIWrapper(
        top_k_results=3,
        doc_content_chars_max=4000
    )
    
    #2. Initialize the Arxiv Query Object
    arxiv = ArxivQueryRun(api_wrapper=arxiv_api_wrapper)

    #3. Invoke the Arxiv Query Object
    response = arxiv.invoke(query)
    return response

tool_arxiv_search.invoke("What are the latest papers on AI?")

In [ ]:
@tool
def personal_info(query: str) -> str:
    """Use this tool when you need to answer questions about personal information"""
    
    infos = [
        {
            "name": "Ojas Dighe",
            "age": 25,
            "location": "India",
            "interests": "coding, building things, reading, writing"
        },
        {
            "name": "John Doe",
            "age": 30,
            "location": "USA",
            "interests": "coding, building things, reading, writing"
        },
        {
            "name": "Jane Smith",
            "age": 28,
            "location": "Canada",
            "interests": "coding, building things, reading, writing"
        }
    ]

    for info in infos:
        if info["name"] == query:
            return f"Name: {info['name']}, Age: {info['age']}, Location: {info['location']}, Interests: {info['interests']}"


personal_info.invoke("Ojas Dighe")

'Name: Ojas Dighe, Age: 25, Location: India, Interests: coding, building things, reading, writing'

## Bind Tools

In [ ]:
toolkit = [tool_duckduckgo_search, tool_wikipedia_search, tool_arxiv_search, personal_info]

# Step 1: Tool Binding
llm_bind = llm.bind_tools(toolkit)

llm_bind.invoke("What is the capital of France?")

AIMessage(content='The capital of France is Paris.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 223, 'total_tokens': 303, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DUAzV1BGqyhMdsq0hLAuvsQGcIdkN', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d86e2-2495-7541-bd8f-23009a26408f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 223, 'output_tokens': 80, 'total_tokens': 303, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 64}})

In [ ]:
# We will not be able to get content in LLM response even after tool binding. 
# You will see tool calls in the response, but no content in AIMessage Object.
# To solve this, we need to use a tool calling agent. Which is created in ReAct_Agent.ipynb
llm_bind.invoke("Tell me about Ojas Dighe. Make tool calls if necessary")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 229, 'total_tokens': 259, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DUAzXXIxvnCwbkQcNjCnFoMaQlUDa', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d86e2-2f78-7401-a25c-84a8467fdae7-0', tool_calls=[{'name': 'tool_duckduckgo_search', 'args': {'query': 'Ojas Dighe'}, 'id': 'call_1qXdq6APOMxsZjgoySZzMwjs', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 229, 'output_tokens': 30, 'total_tokens': 259, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## Notes: Tool Definition & Binding

### What Is Tool Binding?

Tool binding is the process of **registering external functions with an LLM** so the model knows what tools are available and how to invoke them. The LLM itself cannot execute code — it can only output a structured request (JSON) saying "call this function with these arguments." The runtime (your code or an agent framework) is responsible for actually executing the function and feeding the result back.

### The `@tool` Decorator (LangChain)

```python
@tool
def my_tool(query: str) -> str:
    """Docstring becomes the tool description the LLM sees."""
    ...
```

The `@tool` decorator does three things:
1. **Wraps** the function as a LangChain `Tool` object.
2. **Extracts the JSON schema** from the function signature (parameter names, types, defaults).
3. **Uses the docstring** as the tool's `description` field — this is what the LLM reads to decide when to use the tool.

### How `bind_tools()` Works Under the Hood

```
toolkit = [tool_duckduckgo_search, tool_wikipedia_search, ...]
llm_bind = llm.bind_tools(toolkit)
```

When you call `bind_tools(toolkit)`, LangChain:
1. Serializes each tool into an OpenAI-compatible **function schema** (name, description, parameters as JSON Schema).
2. Attaches these schemas to **every subsequent API request** as the `tools` parameter.
3. The LLM sees these schemas in its context window alongside the user message.

The serialized schema for a tool looks like:
```json
{
  "type": "function",
  "function": {
    "name": "tool_duckduckgo_search",
    "description": "Use this tool to gather information about current events or general knowledge",
    "parameters": {
      "type": "object",
      "properties": {
        "query": {"type": "string"}
      },
      "required": ["query"]
    }
  }
}
```

### How the LLM Decides: Memory vs Tool Call

On every prompt, the model evaluates:

| Condition | Model behavior | Example |
|---|---|---|
| Answer is in training data and no tool matches better | Responds directly, `tool_calls=[]` | "What is the capital of France?" → "Paris" |
| A tool's description matches the query | Emits `tool_call` JSON with function name + args | "Tell me about Ojas Dighe" → calls `personal_info` |
| Multiple tools might help | May emit multiple parallel tool calls | Search + Wikipedia simultaneously |
| Uncertain | May attempt a tool call even if it could answer from memory | Depends on docstring specificity |

**The docstring is the routing mechanism.** Vague descriptions like "A useful tool" cause poor routing. Specific descriptions like "Use this tool when you need to answer questions about personal information" act as targeted routing instructions.

### The Limitation: Binding ≠ Execution

This is the critical insight demonstrated in Cell 9:

```
llm_bind.invoke("Tell me about Ojas Dighe")
→ AIMessage(content='', tool_calls=[{name: 'personal_info', args: {query: 'Ojas Dighe'}}])
```

The model correctly identifies which tool to call and with what arguments, but:
- The `content` field is **empty** — the model defers to the tool.
- The tool call is a **request**, not an execution. Nobody runs `personal_info("Ojas Dighe")`.
- The response is an `AIMessage` with `tool_calls` populated but no actual answer.

To close the loop, you need an **agent** (ReAct loop) that:
1. Receives the `AIMessage` with `tool_calls`
2. Executes the corresponding Python functions
3. Sends the results back as `ToolMessage`s
4. Lets the LLM generate a final answer incorporating the tool results

This is exactly what `2_ReAct_Agent.ipynb` builds.

### Tools Defined in This Notebook

| Tool | Source | Docstring purpose |
|---|---|---|
| `tool_duckduckgo_search` | DuckDuckGo API | Current events, general knowledge |
| `tool_wikipedia_search` | Wikipedia API | Historical events |
| `tool_arxiv_search` | arXiv API | Academic papers |
| `personal_info` | In-memory list | Personal information lookup |

### Key Takeaways

1. **Tool calling is an output format, not a capability.** The LLM was fine-tuned (via RLHF) to emit structured JSON when it recognizes tool schemas in the prompt.
2. **Docstrings are critical.** They are the only signal the model uses to route between tools.
3. **`bind_tools()` alone is insufficient** for end-to-end tool use. You also need an agent loop to execute the calls and feed results back.
4. **Type hints matter.** LangChain uses them to generate the JSON schema the model sees. Wrong types = wrong invocations.

### Sources
- [LangChain Tools Documentation](https://python.langchain.com/docs/concepts/tools/)
- [OpenAI Function Calling Docs](https://platform.openai.com/docs/guides/function-calling)